## MATH 4511 Assignment 1


### Exercise 1

In [2]:
# Exercise 1
T = 5
F = 1200
c = 0.05
m = 2
y = 0.04 # annual effective yield rate of the bond


#a) The current price of the bond
def initial_price(T, F, c, m, y):
    N = int(round(T*m))
    coupons = F*c/m
    P0 = sum(
        coupons/(1+y)**(t/m) for t in range(1, N+1)
    ) + F/(1+y)**(T)
    return P0
print(f'The current price of the bond is ${initial_price(T, F, c, m, y):.2f}')

The current price of the bond is $1256.07


**Modified duration**

$$
D_{Mac}(i) = \frac{\sum_k t_k\, C_k (1+i)^{-t_k}}{\sum_k C_k (1+i)^{-t_k}}, \qquad
D_{Mod}(i) = -\frac{P'(i)}{P(i)} = \frac{D_{Mac}(i)}{1+i}
$$

**Convexity**

$$
C_{Mod}(i) = \frac{P''(i)}{P(i)} = \frac{\sum_k t_k (t_k+1)\, C_k (1+i)^{-t_k-2}}{P(i)}
$$

**Price change for a yield shift $\varepsilon$**

$$
\frac{P(i_0+\varepsilon) - P(i_0)}{P(i_0)} \approx -\varepsilon\, D_{Mod} + \frac{\varepsilon^2}{2}\, C_{Mod}
$$


In [3]:
def d_mac(T, F, c, m, y): # Mac
    coupons = F*c/m
    N = int(round(T*m))
    temp_sum = F*T*(1+y)**(-T)
    for i in range(1, 1+N):
        temp_sum += (i/m)*coupons*(1+y)**(-i/m)
    P0 = initial_price(T, F, c, m, y)
    return (temp_sum / P0)

def d_mod(T, F, c, m, y): # Modified duration
    mac_duration = d_mac(T, F, c, m, y)
    return (mac_duration/(1+y))

def c_mod(T, F, c, m, y):
    coupons = F*c/m
    N = int(round(T*m))
    temp_sum = F*T*(T+1)*(1+y)**(-T-2)
    for i in range(1, N+1):
        temp_sum += coupons*(i/m)*(i/m+1)*(1+y)**(-i/m-2)
    P0 = initial_price(T, F, c, m, y)
    return (temp_sum/P0)

print(f'The modified duration of the bond is {d_mod(T, F, c, m, y):.2f}\nThe modified convexity of the bond is {c_mod(T, F, c, m, y):.2f}')

The modified duration of the bond is 4.33
The modified convexity of the bond is 24.13


In [4]:
#c) Calculate the percentage price in bond price using second order approximation if the annual effective yield of the bond decreases by 0.1%
def second_ord_approx(T, F, c, m, y, eps):
    mod_duration = d_mod(T, F, c, m, y)
    mod_convexity = c_mod(T, F, c, m, y)
    price_change_percentage = -eps*mod_duration + eps**(2)*mod_convexity/2
    return price_change_percentage
print(f'The price change of the bond when the annual effective yield decreases with 0.1% is {second_ord_approx(T, F, c, m, y, eps=-0.001)*100:.2f}% using the second order approximation')

The price change of the bond when the annual effective yield decreases with 0.1% is 0.43% using the second order approximation


### Exercise 2

In [5]:
import numpy as np
from scipy.optimize import brentq
spot_rates = { #the spot rates are quoted as annual effective rates
    0.25: 0.021,
    0.50: 0.021,
    0.75: 0.022,
    1.00: 0.024,
    1.25: 0.024,
    1.50: 0.024,
    1.75: 0.026,
    2.00: 0.026,
    2.25: 0.029,
    2.50: 0.029,
    2.75: 0.029,
    3.00: 0.031,
}

T = 3
F = 1000
c = 0.04
m = 2

def B(t, spot_rates): #We assume that t is denoted in years
    discount_rate = 1/(1+spot_rates[t])**(t)
    return discount_rate

# The initial price is the sum of the present value of the cash flows
def initial_price_spot(T, F, c, m, spot_rates): #This is used when the discount rate for the yields are not fixed
    coupons = F*c/m
    N = int(round(T*m))
    temp_sum = F*B(T, spot_rates)
    time = 0
    for t in range(1, N+1):
        temp_sum += coupons*B(t/m, spot_rates)
    return temp_sum

print(f'The intial price of the bond is ${initial_price_spot(T, F, c, m, spot_rates):.2f}')

# The yield of the bond is the discount rate which makes the npv of the bond equal to 0
# This requires solving a ploynomial of degree N. We choose to solve it numerically

def bond_yield(T, F, c, m, spot_rates, tol=1e-4):
    P0 = initial_price_spot(T, F, c, m, spot_rates)
    yield_numerical = brentq(lambda y: initial_price(T, F, c, m, y) - P0, 0.0, 0.5) # The last values are the bounds for the yields we consider
    return yield_numerical
print(f'The annual effective yield of the bond is {bond_yield(T, F, c, m, spot_rates)*100:.4f}%')

def effective_to_nominal(y, m):
    yield_nominal = m*((1+y)**(1/m)-1)
    return yield_nominal
print(f'The annual nominal yield of the bond is {effective_to_nominal(bond_yield(T, F, c, m, spot_rates), m)*100:.4f}%')




The intial price of the bond is $1026.98
The annual effective yield of the bond is 3.0754%
The annual nominal yield of the bond is 3.0522%
